# Skin Lesion Classification — PAD-UFES-20
**Week 2 deliverable:** annotated dataset + classification code + metrics.

Project 1 (Patient Skin Lesion Monitoring), Computer Vision Semester 6.  
Model: **ResNet50** with transfer learning (ImageNet pretrained).

## How to use (read this first)
1. Runtime menu -> Change runtime type -> Hardware accelerator -> **GPU (T4)** -> Save.
2. In your Google Drive, inside the folder `skin-lesion-monitoring-cv_dataset`,
   place BOTH files:
   - `images.zip`  (contains imgs_part_1/2/3)
   - `metadata.csv` (label file)
3. Run cells **top to bottom**. Edit `DATASET_DIR` in the Config cell only if the
   Drive folder name differs.

If you see `Transport endpoint is not connected`, just re-run the Drive-mount
cell (it force-remounts) and continue from there.

In [ ]:
# 1. Imports + GPU check
import os, glob, zipfile, time, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime -> Change runtime type -> GPU (T4).')

In [ ]:
# 2. Mount Google Drive (force_remount fixes 'Transport endpoint' errors)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 3. Config -- edit DATASET_DIR if your Drive folder name differs
DATASET_DIR = '/content/drive/MyDrive/skin-lesion-monitoring-cv_dataset'
ZIP_ON_DRIVE  = os.path.join(DATASET_DIR, 'images.zip')
META_ON_DRIVE = os.path.join(DATASET_DIR, 'metadata.csv')
DATA_ROOT = '/content/pad-ufes-20'               # zip extracted here (fast local disk)
OUT_DIR   = '/content/drive/MyDrive/pad-ufes-20-results'

IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 10
LR         = 1e-4
VAL_SPLIT  = 0.2
SEED       = 42
CLASSES    = ['BCC', 'SCC', 'ACK', 'SEK', 'MEL', 'NEV']

torch.manual_seed(SEED); np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isfile(ZIP_ON_DRIVE),  f'Zip not found: {ZIP_ON_DRIVE}'
assert os.path.isfile(META_ON_DRIVE), f'metadata.csv not found: {META_ON_DRIVE}'
print('Zip OK :', ZIP_ON_DRIVE)
print('Meta OK:', META_ON_DRIVE)

In [ ]:
# 4. Copy zip + metadata to LOCAL disk first (robust), then unzip locally.
# Reading a big zip directly off the Drive mount causes
# 'Transport endpoint is not connected' -- copying first avoids that.
local_zip  = '/content/images.zip'
local_meta = '/content/metadata.csv'

t = time.time()
shutil.copy(ZIP_ON_DRIVE, local_zip)
shutil.copy(META_ON_DRIVE, local_meta)
print(f'Copied zip+csv to local disk in {time.time() - t:.0f}s '
      f'({os.path.getsize(local_zip) / 1e6:.0f} MB zip)')

t = time.time()
os.makedirs(DATA_ROOT, exist_ok=True)
with zipfile.ZipFile(local_zip) as z:
    z.extractall(DATA_ROOT)
shutil.copy(local_meta, os.path.join(DATA_ROOT, 'metadata.csv'))
print(f'Extracted in {time.time() - t:.0f}s into {DATA_ROOT}')

n_img = sum(len(glob.glob(os.path.join(DATA_ROOT, '**', e), recursive=True))
            for e in ('*.png', '*.jpg', '*.jpeg'))
print('Images found after extract:', n_img)

In [ ]:
# 5. Build the annotated dataset (image path -> class label)
# Recursively index images so the imgs_part_1/2/3 folders are handled automatically.
paths = {}
for ext in ('*.png', '*.jpg', '*.jpeg'):
    for p in glob.glob(os.path.join(DATA_ROOT, '**', ext), recursive=True):
        paths[os.path.basename(p)] = p
print('Indexed images:', len(paths))

meta = pd.read_csv(os.path.join(DATA_ROOT, 'metadata.csv'))
meta = meta[['img_id', 'diagnostic']].dropna()
meta = meta[meta['diagnostic'].isin(CLASSES)].copy()
meta['path'] = meta['img_id'].map(paths)
meta = meta.dropna(subset=['path']).reset_index(drop=True)
meta['label'] = meta['diagnostic'].map({c: i for i, c in enumerate(CLASSES)})

print('Usable samples:', len(meta))
print(meta['diagnostic'].value_counts())
meta.to_csv(os.path.join(OUT_DIR, 'annotated_dataset.csv'), index=False)

In [ ]:
# 6. Dataset, transforms, stratified train/val split
train_df, val_df = train_test_split(
    meta, test_size=VAL_SPLIT, stratify=meta['label'], random_state=SEED)
print('Train:', len(train_df), ' Val:', len(val_df))

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(), norm])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), norm])

class LesionDS(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r['path']).convert('RGB')
        return self.tf(img), int(r['label'])

train_dl = DataLoader(LesionDS(train_df, train_tf), batch_size=BATCH_SIZE,
                      shuffle=True, num_workers=2)
val_dl = DataLoader(LesionDS(val_df, val_tf), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=2)

In [ ]:
# 7. Model: ResNet50 transfer learning + class weights for imbalance
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
model = model.to(device)

counts = train_df['label'].value_counts().sort_index().values
weights = torch.tensor(counts.sum() / (len(counts) * counts),
                       dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [ ]:
# 8. Training loop (saves best model by val accuracy)
best_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train(); tr_loss = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        tr_loss += loss.item() * x.size(0)
    tr_loss /= len(train_dl.dataset)

    model.eval(); correct = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
    val_acc = correct / len(val_dl.dataset)
    print(f'Epoch {epoch:2d}/{EPOCHS}  train_loss={tr_loss:.4f}  val_acc={val_acc:.4f}')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(OUT_DIR, 'best_model.pth'))
print('Best val accuracy:', round(best_acc, 4))

In [ ]:
# 9. Metrics: classification report + confusion matrix
model.load_state_dict(torch.load(os.path.join(OUT_DIR, 'best_model.pth')))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in val_dl:
        x = x.to(device)
        y_pred += model(x).argmax(1).cpu().tolist()
        y_true += y.tolist()

report = classification_report(y_true, y_pred, target_names=CLASSES, digits=4)
print(report)
with open(os.path.join(OUT_DIR, 'metrics.txt'), 'w') as f:
    f.write('Model: ResNet50 (transfer learning)\n')
    f.write('Best val accuracy: %.4f\n\n' % best_acc)
    f.write(report)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES)
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion Matrix')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
fig.colorbar(im); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion_matrix.png'), dpi=120)
plt.show()
print('Saved model + metrics + confusion matrix to', OUT_DIR)